# 01 · Dataset

Genera los ejemplos sintéticos, valida todo el dataset y arma los splits estratificados por `prioridad`. Los ejemplos `manual`/`real` los escribe el equipo a mano.

> **Ojo:** los splits ya están generados y versionados en `data/splits/` (incluyen los 300 ejemplos manuales). No hace falta correr este notebook antes de los experimentos; si se corre, regenera los sintéticos y los splits.

In [ ]:
# === Setup ===
# En Google Colab: clona la rama del proyecto, instala el paquete y monta Drive.
# Fuera de Colab (local, desde la raíz del repo) esta celda no hace nada.
import os, sys
EN_COLAB = "google.colab" in sys.modules
RAMA = "splits-con-manual"  # cambiar a "main" cuando la rama se fusione
if EN_COLAB:
    if not os.path.exists("/content/triaje-clinico-lora"):
        !git clone -q -b {RAMA} https://github.com/Hammonne/triaje-clinico-lora.git /content/triaje-clinico-lora
    %cd /content/triaje-clinico-lora
    !git pull -q
    # Colab trae torchao 0.10, que peft 0.21 rechaza (pide >= 0.16); no lo usamos.
    !pip uninstall -q -y torchao
    # Colab ya trae las dependencias: esto solo verifica versiones mínimas e instala `triaje`.
    !pip install -q -r requirements.txt
    !pip install -q -e .
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# === Celda de configuración (única fuente de parámetros) ===
import sys; sys.path.insert(0, "src")
from pathlib import Path
from triaje.config import Config, set_seed

config = Config()
# Lee los resultados versionados en el repositorio (results/). Para analizar los de
# una corrida nueva guardada en Drive, apuntar ahí:
# config.results_dir = Path("/content/drive/MyDrive/triaje/results")
set_seed(config.seed)
print("modelo:", config.model_name, "| resultados en:", config.results_dir)

## Generar ejemplos sintéticos

Solo `origen: sintetico`. El 20% manual lo escribe el equipo.

In [ ]:
!python scripts/generar_sintetico.py --n 1200 --seed 42 --salida data/sintetico/lote01.jsonl

## Validar todo el dataset (manual + real + sintetico)

In [ ]:
!python scripts/validar_dataset.py data/sintetico/*.jsonl data/manual/*.jsonl

## Construir y guardar los splits (estratificados, semilla fija)

In [ ]:
from triaje.data import cargar_todos, dividir, guardar_splits
ejemplos = cargar_todos(config.data_dir)
splits = dividir(ejemplos, config.frac_train, config.frac_val, config.frac_test, seed=config.seed_split)
print(splits.resumen())
guardar_splits(splits, config.data_dir / "splits")